# Step 3: Deploy AutoGluon Model to SageMaker Endpoint

Deploys a real-time endpoint using the AutoGluon DLC inference container (SDK v3).

The v3 SDK uses a resource-based API: `Model.create()` → `EndpointConfig.create()` → `Endpoint.create()` → `Endpoint.invoke()`.

## Configuration

In [ ]:
import boto3
import sagemaker
import time
import os

REGION = boto3.session.Session().region_name
sess = sagemaker.session.Session()
BUCKET = sess.default_bucket()
S3_PREFIX = "autogluon-tabular"

AG_VERSION = "1.5"
PY_VERSION = "py312"
INSTANCE_TYPE = "ml.m5.xlarge"

# After running the pipeline (3-pipeline/pipeline.ipynb), find the training job output:
#   aws s3 ls s3://{BUCKET}/{S3_PREFIX}/pipeline/model/ --recursive
# Update the path below with your pipeline's training output
PIPELINE_MODEL = f"s3://{BUCKET}/{S3_PREFIX}/pipeline/model/<your-pipeline-execution>/output/model.tar.gz"

# Repacked model location (with code/inference.py)
MODEL_DATA = f"s3://{BUCKET}/{S3_PREFIX}/inference/model.tar.gz"

# Inference script to package (serve.py is in the same directory as this notebook)
SERVE_SCRIPT = os.path.abspath("serve.py")

# Resource names (timestamp-based to avoid conflicts)
ts = int(time.time())
MODEL_NAME = f"ag-tabular-{ts}"
ENDPOINT_CONFIG_NAME = f"ag-tabular-config-{ts}"
ENDPOINT_NAME = f"ag-tabular-endpoint-{ts}"

# IAM role
from sagemaker.core.helper.session_helper import get_execution_role
ROLE_ARN = get_execution_role()

print(f"Region:         {REGION}")
print(f"Pipeline model: {PIPELINE_MODEL}")
print(f"Repacked model: {MODEL_DATA}")
print(f"Endpoint:       {ENDPOINT_NAME}")

## Resolve Inference Image URI

In [ ]:
from sagemaker.core import image_uris

image_uri = image_uris.retrieve(
    "autogluon",
    region=REGION,
    version=AG_VERSION,
    py_version=PY_VERSION,
    image_scope="inference",
    instance_type=INSTANCE_TYPE,
)
print(f"Inference image: {image_uri}")

## Repackage Model with Inference Code

The AutoGluon DLC uses TorchServe and expects `code/inference.py` inside the model archive.
Pipeline training outputs only contain model artifacts (pkl files), so we repackage with our `serve.py`.

In [ ]:
import subprocess, tempfile, shutil, tarfile

with tempfile.TemporaryDirectory() as tmpdir:
    # Download original model
    local_tar = os.path.join(tmpdir, "model.tar.gz")
    subprocess.run(["aws", "s3", "cp", PIPELINE_MODEL, local_tar], check=True)

    # Extract
    extract_dir = os.path.join(tmpdir, "model")
    os.makedirs(extract_dir)
    with tarfile.open(local_tar) as tf:
        tf.extractall(extract_dir, filter="data")

    # Add inference code
    code_dir = os.path.join(extract_dir, "code")
    os.makedirs(code_dir, exist_ok=True)
    shutil.copy2(SERVE_SCRIPT, os.path.join(code_dir, "inference.py"))
    print(f"Added code/inference.py from {SERVE_SCRIPT}")

    # Repack
    repack_tar = os.path.join(tmpdir, "model-repack.tar.gz")
    with tarfile.open(repack_tar, "w:gz") as tf:
        for item in os.listdir(extract_dir):
            tf.add(os.path.join(extract_dir, item), arcname=item)

    # Upload
    subprocess.run(["aws", "s3", "cp", repack_tar, MODEL_DATA], check=True)
    print(f"Repacked model uploaded to {MODEL_DATA}")

## Create Model, EndpointConfig, and Endpoint (v3 resource API)

In [ ]:
from sagemaker.core.resources import Model, EndpointConfig, Endpoint
from sagemaker.core.shapes.shapes import ContainerDefinition, ProductionVariant

# 1. Create Model
model = Model.create(
    model_name=MODEL_NAME,
    primary_container=ContainerDefinition(
        image=image_uri,
        model_data_url=MODEL_DATA,
    ),
    execution_role_arn=ROLE_ARN,
)
print(f"Model created: {MODEL_NAME}")

# 2. Create EndpointConfig
endpoint_config = EndpointConfig.create(
    endpoint_config_name=ENDPOINT_CONFIG_NAME,
    production_variants=[
        ProductionVariant(
            variant_name="AllTraffic",
            model_name=MODEL_NAME,
            initial_instance_count=1,
            instance_type=INSTANCE_TYPE,
        )
    ],
)
print(f"EndpointConfig created: {ENDPOINT_CONFIG_NAME}")

# 3. Create Endpoint
endpoint = Endpoint.create(
    endpoint_name=ENDPOINT_NAME,
    endpoint_config_name=ENDPOINT_CONFIG_NAME,
)
print(f"Endpoint creating: {ENDPOINT_NAME}")

# 4. Wait for InService
endpoint.wait_for_status("InService")
print(f"Endpoint ready: {ENDPOINT_NAME}")

## Test the endpoint with a sample prediction

In [ ]:
import json
import pandas as pd

sample_data = pd.DataFrame({
    "age": [39], "workclass": ["State-gov"], "fnlwgt": [77516],
    "education": ["Bachelors"], "education-num": [13],
    "marital-status": ["Never-married"], "occupation": ["Adm-clerical"],
    "relationship": ["Not-in-family"], "race": ["White"], "sex": ["Male"],
    "capital-gain": [2174], "capital-loss": [0],
    "hours-per-week": [40], "native-country": ["United-States"],
})
csv_payload = sample_data.to_csv(index=False)

response = endpoint.invoke(body=csv_payload, content_type="text/csv", accept="application/json")
# response.body is a botocore StreamingBody
result = json.loads(response.body.read().decode("utf-8"))
print(f"Prediction: {json.dumps(result, indent=2)}")

## Cleanup

Uncomment the cell below to delete the endpoint when you are done.

In [ ]:
# endpoint.delete()
# endpoint_config.delete()
# model.delete()
# print("Endpoint, config, and model deleted.")